# Mesh 6 · From a surface to a structure: subdivide, pattern, thicken

**Application.** Companion to [Week 05 §9](../README.md#9-changing-a-mesh) and
the scripts `mesh_basics/13_subdivision_schemes.py` to `16_conway_chained.py`.
Uses notebook 04.

![One shell patterned three ways: quads, triangles, and the dual of the triangles, with the nodes, bars and metres of each](../figures/app-patterns.svg)

[`04_changing_a_mesh`](04_changing_a_mesh.ipynb) showed the operators on a
cube and a grid. Here the same operators answer a design question: **one
shell, three ways to build it. What does each cost?**

In [ ]:
# Setup: run this cell first. It finds the shared drawing helper (viz.py),
# whether this notebook sits in Lecture/notebooks/ or is your copy in MyWork/.
import sys
import pathlib

HERE = pathlib.Path.cwd()
NB = next(p / "Lecture" / "notebooks" for p in [HERE, *HERE.parents]
          if (p / "Lecture" / "notebooks" / "viz.py").exists())
sys.path.insert(0, str(NB))
from viz import chart, draw_graph, draw_mesh, draw_points, ramp, row, view3d

from collections import Counter

from compas.datastructures import Mesh
from compas.geometry import distance_point_plane

## Step 1 — a cage: sixteen points you can design with
A 12 m square, 3 by 3 faces. The boundary stays on the ground and the four
inner vertices are lifted. This coarse mesh is the **cage**: it is not the
shell, it controls the shell.

In [ ]:
cage = Mesh.from_meshgrid(dx=12, nx=3, dy=12, ny=3)
for v in cage.vertices():
    cage.vertex_attribute(v, "z", 0.0 if cage.is_vertex_on_boundary(v) else 4.0)

inner = [v for v in cage.vertices() if not cage.is_vertex_on_boundary(v)]
print("cage: V E F =", cage.number_of_vertices(), cage.number_of_edges(), cage.number_of_faces())
print("lifted vertices:", inner)
draw_mesh(cage, "the cage", view="iso")

## Step 2 — subdivision turns the cage into a surface
Catmull–Clark splits every face into four and averages the positions. Each
level multiplies the face count by four and makes the surface smoother.
Watch the counts, and watch the top: averaging pulls the surface inside the
cage, so the shell ends up lower than the points you lifted.

In [ ]:
levels = [cage] + [cage.subdivided(k=k, scheme="catmullclark") for k in (1, 2)]
for k, m in enumerate(levels):
    print(f"level {k}: V {m.number_of_vertices():3d}  E {m.number_of_edges():3d}  "
          f"F {m.number_of_faces():3d}   top at z = {max(m.vertices_attribute('z')):.2f} m")
row(*[draw_mesh(m, f"level {k}", view="iso", vertex_labels=False, size=(330, 280))
      for k, m in enumerate(levels)])

### The cage is the design handle
Move **one** cage vertex and subdivide again: the whole surface follows.
Sixteen numbers to design with, instead of 169.

In [ ]:
pushed = cage.copy()
pushed.vertex_attribute(inner[0], "z", 7.0)
shell = levels[2]
row(draw_mesh(shell, "the shell", view="iso", vertex_labels=False, size=(440, 320)),
    draw_mesh(pushed.subdivided(k=2, scheme="catmullclark"), f"cage vertex {inner[0]} raised to 7 m",
              view="iso", vertex_labels=False, size=(440, 320)))

In [ ]:
view3d((shell, {"show_edges": True}))

## Step 3 — three patterns on the same surface
The shell will be built from bars and panels. Same surface, three meshes:

- **quads**: the subdivided mesh as it is
- **triangles**: every quad split by a diagonal
- **the dual** of the triangles: one new vertex per triangle, one new face
  per old vertex

Before running the next cell, predict the dual from the triangle mesh's
valences below: *a vertex where n bars meet becomes a face with n sides.*

In [ ]:
quads = shell
triangles = shell.copy()
triangles.quads_to_triangles()
print("triangle mesh, bars per node:",
      dict(sorted(Counter(triangles.vertex_degree(v) for v in triangles.vertices()
                          if not triangles.is_vertex_on_boundary(v)).items())))

In [ ]:
dual = triangles.dual()
print("dual mesh, sides per panel:  ",
      dict(sorted(Counter(len(dual.face_vertices(f)) for f in dual.faces()).items())))
row(draw_mesh(quads, "quads", vertex_labels=False, size=(330, 330)),
    draw_mesh(triangles, "triangles", vertex_labels=False, size=(330, 330)),
    draw_mesh(dual, "dual of the triangles", vertex_labels=False, size=(330, 330)))

Not a honeycomb. `quads_to_triangles` cuts each quad from its own first
vertex, and after subdivision neighbouring faces start at different corners,
so the diagonals alternate. Inner nodes end up with 8 or 4 bars, and the dual
is made of **octagons and squares**. A honeycomb needs 6 bars at every node
(notebook 04 §6 got one, from a plain grid). The valences told you before you
drew anything.

## Step 4 — what each pattern costs
Count the things a builder pays for: nodes, bars, metres of bar, the most
bars meeting in one node, and how far the worst panel is from flat.

In [ ]:
def worst_warp(mesh):
    """Largest distance from a face's own vertices to its best-fit plane."""
    return max(abs(distance_point_plane(mesh.vertex_coordinates(v), mesh.face_plane(f)))
               for f in mesh.faces() for v in mesh.face_vertices(f))


print(f"{'pattern':10s} {'nodes':>6s} {'bars':>6s} {'panels':>7s} {'bar m':>7s} {'max bars/node':>14s} {'worst warp':>11s}")
for name, m in (("quads", quads), ("triangles", triangles), ("dual", dual)):
    metres = sum(m.edge_length(e) for e in m.edges())
    busiest = max(m.vertex_degree(v) for v in m.vertices())
    print(f"{name:10s} {m.number_of_vertices():6d} {m.number_of_edges():6d} {m.number_of_faces():7d} "
          f"{metres:7.1f} {busiest:14d} {worst_warp(m) * 1000:8.0f} mm")

Read the table as a trade:

- **Triangles** are the only pattern with flat panels (three points always
  share a plane) and the only one that is rigid as a pin-jointed frame (the
  truss notebook, `08_mesh_as_truss`, shows why). They pay for it with the
  most bars and with 8 bars in one node.
- **The dual** has the simplest nodes (3 bars, everywhere) and the fewest
  metres of bar. It needs the most nodes, and its octagons are not flat.
- **Quads** need the fewest nodes, with 4 bars each. Their panels warp, and
  a quad frame is not rigid on its own, which is why quad gridshells carry
  extra bracing.

## Step 5 — give it thickness
A surface has no volume. `thickened` offsets it both ways and closes the
rim, which makes a solid you can weigh, print or send to a slicer.

In [ ]:
thickness = 0.12
solid = shell.thickened(thickness)
print("surface: closed?", shell.is_closed(), "| euler", shell.euler())
print("solid:   closed?", solid.is_closed(), "| euler", solid.euler())
volume = solid.volume()
print(f"volume {volume:.2f} m3   (area x thickness = {shell.area() * thickness:.2f} m3)")
print(f"as concrete at 2400 kg/m3: {volume * 2400 / 1000:.1f} tonnes")

In [ ]:
view3d((solid, {"show_edges": False}))

Euler went from 1 (a disc) to 2 (a closed ball): the check from Week 05 §8
confirms the solid has no gap along the rim.

## Your turn
1. Raise the four inner cage vertices to 6 m. Which column of the Step 4
   table changes most?
2. Use `scheme="quad"` instead of `"catmullclark"` in Step 2. The counts are
   the same; what happened to the shape, and why?
3. Build a real honeycomb on this shell: every diagonal must run the same
   way. Start from a plain grid (`Mesh.from_meshgrid(dx=12, nx=12)`), lift
   its vertices with a formula of your own, triangulate, and check the
   valences before you take the dual.
4. `mesh_conway_ambo` (from `compas.datastructures`) cuts every corner at the
   middle of its edges. Add it as a fourth row of the table.